# LBM-CaImAn full pipeline

Runs the CaImAn pipeline on every-other z-plane of `D:/demo/raw` and writes suite2p-compatible outputs to `D:/demo/caiman-python/`.

Output layout per plane (mirrors `lbm_suite2p_python.pipeline()`):
`data_raw.bin`, `data.bin`, `ops.npy`, `stat.npy`, `iscell.npy`, `F.npy`, `Fneu.npy`, `spks.npy`, `dff.npy`, `roi_stats.npy`, plus diagnostic figures (`01_…15_*.png`).

In [ ]:
from pathlib import Path

import numpy as np
from mbo_utilities import imread

import lbm_caiman_python as lcp

print(f"lbm_caiman_python: {lcp.__version__}")

## 1. Paths

In [ ]:
input_path = Path(r"D:/demo/raw")
output_path = Path(r"D:/demo/caiman-python")
output_path.mkdir(parents=True, exist_ok=True)

print(f"Input:  {input_path}")
print(f"Output: {output_path}")

## 2. Inspect input

Open the raw volume lazily and read its metadata so we know how many z-planes are available before launching the pipeline.

In [ ]:
data = imread(input_path)
meta = dict(data.metadata or {})

num_planes = getattr(data, "num_planes", None)
if num_planes is None:
    num_planes = int(data.shape5d[2]) if hasattr(data, "shape5d") else 1

print(f"Shape: {data.shape}, dtype: {data.dtype}")
print(f"Num planes: {num_planes}")
print(f"Frame rate: {meta.get('fs', meta.get('fr', 'N/A'))} Hz")
print(f"Pixel size: dx={meta.get('dx', 'N/A')} um, dy={meta.get('dy', 'N/A')} um")

## 3. Pick every-other z-plane

1-based indexing matches `lbm_caiman_python` / `lbm_suite2p_python`: `[1, 3, 5, ...]` picks every odd plane. Flip to `[2, 4, 6, ...]` for the even subset.

In [ ]:
planes = list(range(1, num_planes + 1, 2))
print(f"Processing {len(planes)} planes (of {num_planes}): {planes}")

## 4. CaImAn parameters

Start from `lcp.default_ops()` and tune for the LBM acquisition. Pixel size sets `gSig` / `gSiz`; patch grid is sized for ~10% cell density per patch.

In [ ]:
ops = lcp.default_ops()

fs = float(meta.get("fs", meta.get("fr", 17.0)))
dx = float(meta.get("dx", 2.0))
dy = float(meta.get("dy", dx))

ops["fs"] = fs
ops["fr"] = fs
ops["dxy"] = (dy, dx)
ops["decay_time"] = 0.7

# neuron size from 10-20 um target
neuron_min_um, neuron_max_um = 10, 20
half_neuron_px = max(1, int(round((neuron_min_um / 2) / dx)))
gSiz_px = int(round(neuron_max_um / dx)) + 1
ops["gSig"] = (half_neuron_px, half_neuron_px)
ops["gSiz"] = (gSiz_px, gSiz_px)

# patch grid
patch_side = 80
patch_area_mm2 = (patch_side * dx * 1e-3) ** 2
neurons_per_patch = int(92000 * patch_area_mm2 * 0.05)
ops["rf"] = patch_side // 2
ops["stride"] = patch_side // 4
ops["K"] = max(neurons_per_patch + 10, 30)

# motion correction + quality
ops["do_motion_correction"] = True
ops["pw_rigid"] = True
ops["merge_thresh"] = 0.85
ops["min_SNR"] = 2.5
ops["rval_thr"] = 0.8

print(f"fs={fs} Hz, dx={dx} um, dy={dy} um")
print(f"gSig={ops['gSig']}, gSiz={ops['gSiz']}, K={ops['K']}")
print(f"rf={ops['rf']}, stride={ops['stride']}")

## 5. Run the pipeline

`lcp.pipeline()` writes `data_raw.bin`, runs CaImAn motion correction into `data.bin`, runs CNMF, synthesizes `stat.npy` / `iscell.npy` / `F.npy` / `Fneu.npy` / `spks.npy` / `dff.npy` / `roi_stats.npy`, and renders the lsp figure set. `force_reg=True` / `force_detect=True` ignore any cached outputs.

In [ ]:
ops_files = lcp.pipeline(
    input_data=data,
    save_path=output_path,
    ops=ops,
    planes=planes,
    keep_raw=False,
    keep_reg=True,
    force_reg=False,
    force_detect=False,
)

print(f"\nProcessed {len(ops_files)} planes")
for f in ops_files:
    print(f"  {f}")

## 6. Inspect results

In [ ]:
summary = []
for ops_file in ops_files:
    plane_dir = Path(ops_file).parent
    o = np.load(ops_file, allow_pickle=True).item()
    iscell = np.load(plane_dir / "iscell.npy", allow_pickle=True)
    n_total = int(iscell.shape[0])
    n_accepted = int(iscell[:, 0].sum())
    summary.append((o.get("plane"), n_accepted, n_total, plane_dir.name))

for plane, n_acc, n_tot, name in summary:
    print(f"plane {plane:2d}: {n_acc:5d} accepted / {n_tot:5d} total  ({name})")

zstats_file = output_path / "zstats.npy"
if zstats_file.exists():
    z = np.load(zstats_file, allow_pickle=True)
    print(f"\nVolume stats saved: {zstats_file}")
    print(f"  total accepted across volume: {int(z['accepted'].sum())}")

In [ ]:
first_plane = Path(ops_files[0]).parent
print(f"Files in {first_plane.name}:")
for f in sorted(first_plane.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<40s} {size_mb:8.2f} MB")